# 10a — Masked training: Vanilla CNN

Retrains on the **masked** clean split (radius-scaled 0.9), matched median seed, so the result is directly comparable to the unmasked model. Only the data path differs from the original training. Saves `<model>_masked_seed<median>.pth` + `_preds.npz`.

**Attach:** 09-build-masked-dataset (the masked images), build-clean-split. GPU on.

In [1]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT = "/kaggle/input/notebooks/tochyokafor/09-build-masked-dataset/masked"   # MASKED images (radius-scaled 0.9)
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"
OUT_DIR   = "/kaggle/working"
SEEDS     = [2025]   # median-accuracy seed, matched to unmasked study
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5
NUM_WORKERS = 2

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f"Missing path: {pth}"
print("Paths OK | seeds:", SEEDS, "| workers:", NUM_WORKERS)

Paths OK | seeds: [2025] | workers: 2


In [2]:
!pip install timm --quiet
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score,
    precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_idx = np.load(f"{SPLIT_DIR}/clean_train_indices.npy").tolist()
val_idx   = np.load(f"{SPLIT_DIR}/clean_val_indices.npy").tolist()
test_idx  = np.load(f"{SPLIT_DIR}/clean_test_indices.npy").tolist()
class_names = open(f"{SPLIT_DIR}/class_names.txt").read().splitlines()
SEVERE_IDX = class_names.index("Severe")
print("Classes:", class_names, "| Severe idx:", SEVERE_IDX)

Device: cuda
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'] | Severe idx: 4


In [3]:
# --- full determinism per run ---
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator(); g.manual_seed(seed)
    return g

In [4]:
train_idx = np.load(f'{SPLIT_DIR}/clean_train_indices.npy')
test_idx  = np.load(f'{SPLIT_DIR}/clean_test_indices.npy')
print('train', len(train_idx), 'test', len(test_idx))

train 2485 test 536


In [5]:
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)), transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20), transforms.ColorJitter(0.3,0.3,0.2,0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, seed, crop_from=None):
    g = set_seed(seed)
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True, num_workers=NUM_WORKERS, generator=g),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=NUM_WORKERS))

In [6]:
def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        run = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,"logits") else out, y)
            loss.backward(); optimizer.step()
            run += loss.item()
        if scheduler: scheduler.step()
        print(f"    epoch {ep+1}/{epochs} loss {run/len(loader):.4f}")
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        out = model(x.to(device))
        out = out.logits if hasattr(out,"logits") else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def save_and_report(name, seed, model, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")
    pr,rc,f1,sup = precision_recall_fscore_support(labels,preds,labels=range(NUM_CLASSES),zero_division=0)
    torch.save(model.state_dict(), f"{OUT_DIR}/{name}_seed{seed}.pth")
    np.savez(f"{OUT_DIR}/{name}_seed{seed}_preds.npz", probs=probs, preds=preds, labels=labels)
    print(f"  [{name} seed {seed}] acc {acc:.4f} | macroF1 {f1m:.4f} | "
          f"SevereRec {rc[SEVERE_IDX]:.3f} | ProlifRec {rc[class_names.index('Proliferate_DR')]:.3f}")
    print(f"  saved {name}_seed{seed}.pth + _preds.npz")

def already_done(name, seed):
    p = f"{OUT_DIR}/{name}_seed{seed}_preds.npz"
    if os.path.exists(p):
        print(f"  [skip] {name} seed {seed} already done"); return True
    return False

In [7]:
import numpy as np, torch
def save_run(model, name, seed, test_loader, size):
    model.eval(); ps=[]; ys=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            xb=xb.to(device); out=model(xb)
            out=out.logits if hasattr(out,'logits') else out
            ps.append(torch.softmax(out,1).cpu().numpy()); ys.append(yb.numpy())
    import numpy as np
    probs=np.concatenate(ps); labels=np.concatenate(ys); preds=probs.argmax(1)
    torch.save(model.state_dict(), f'{OUT_DIR}/{name}_masked_seed{seed}.pth')
    np.savez(f'{OUT_DIR}/{name}_masked_seed{seed}_preds.npz', probs=probs, preds=preds, labels=labels)
    acc=(preds==labels).mean(); print(f'  {name} masked acc={acc:.4f}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device',device)

device cuda


In [8]:
class VanillaCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,3,padding=1); self.conv2=nn.Conv2d(32,64,3,padding=1)
        self.conv3=nn.Conv2d(64,128,3,padding=1); self.conv4=nn.Conv2d(128,256,3,padding=1)
        self.relu=nn.ReLU(); self.maxPool=nn.MaxPool2d(2)
        self.dropout=nn.Dropout(0.25); self.flatten=nn.Flatten()
        self.fc1=nn.Linear(256*14*14,256); self.fc2=nn.Linear(256,num_classes)
    def forward(self,x):
        x=self.relu(self.maxPool(self.conv1(x))); x=self.relu(self.maxPool(self.conv2(x)))
        x=self.relu(self.maxPool(self.conv3(x))); x=self.relu(self.maxPool(self.conv4(x)))
        x=self.flatten(x); x=self.fc1(x); x=self.dropout(x); return self.fc2(x)


In [9]:
s = SEEDS[0]
tr, te = make_loaders(224, s)
set_seed(s)
model = VanillaCNN().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
train_model(model, tr, crit, opt, epochs=EPOCHS)
save_run(model, 'vanilla_cnn', s, te, 224)

    epoch 1/10 loss 0.8893
    epoch 2/10 loss 0.6098
    epoch 3/10 loss 0.4765
    epoch 4/10 loss 0.4369
    epoch 5/10 loss 0.4444
    epoch 6/10 loss 0.3668
    epoch 7/10 loss 0.3346
    epoch 8/10 loss 0.3479
    epoch 9/10 loss 0.3513
    epoch 10/10 loss 0.3288
  vanilla_cnn masked acc=0.6157
